# 🧬 Alpha Factory: Genetic Evolution Engine
This notebook uses Genetic Programming (`gplearn`) to autonomously evolve and discover novel mathematical trading formulas. 
It simulates thousands of generations of 'Survival of the Fittest' to find features that correlate with market-beating returns.

### Updates Included:
- **Fixed Survivorship Bias:** Uses the actual NIFTY 50 Index (`^NSEI`) as the benchmark instead of a static basket mean.
- **Fixed Data Leakage:** Implements a strict Out-Of-Sample (OOS) chronological split to prevent future data from bleeding into training.
- **Out-Of-Sample Validation:** Evaluates the evolved formulas on an untouched dataset to combat Genetic Programming overfitting noise.

In [ ]:
!pip install gplearn pandas numpy yfinance scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from gplearn.genetic import SymbolicTransformer
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("alpha_factory")

In [ ]:
import pandas as pd
import logging

logger.info("Loading strictly stratified universe (Large, Mid, Small Caps)...")
NIFTY_UNIVERSE = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "ICICIBANK.NS", "INFY.NS", "ITC.NS", "SBIN.NS", "BHARTIARTL.NS", "HINDUNILVR.NS", "BAJFINANCE.NS", "LT.NS", "KOTAKBANK.NS", "AXISBANK.NS", "ASIANPAINT.NS", "MARUTI.NS", "SUNPHARMA.NS", "TITAN.NS", "ULTRACEMCO.NS", "TATASTEEL.NS", "NTPC.NS", "POWERGRID.NS", "TATAMOTORS.NS", "M&M.NS", "WIPRO.NS", "NESTLEIND.NS", "HCLTECH.NS", "TECHM.NS", "BAJAJFINSV.NS", "INDUSINDBK.NS", "GRASIM.NS", "HINDALCO.NS", "ONGC.NS", "COALINDIA.NS", "CIPLA.NS", "BRITANNIA.NS", "EICHERMOT.NS", "DIVISLAB.NS", "APOLLOHOSP.NS", "HEROMOTOCO.NS", "BAJAJ-AUTO.NS", "TATASTLLP.NS", "ADANIPORTS.NS", "ADANIENT.NS", "DRREDDY.NS", "TATACONSUM.NS", "UPL.NS", "SBILIFE.NS", "HDFCLIFE.NS", "TRENT.NS", "TVSMOTOR.NS", "CUMMINSIND.NS", "FEDERALBNK.NS", "DIXON.NS", "LUPIN.NS", "BSE.NS", "PERSISTENT.NS", "POLYCAB.NS", "ASTRAL.NS", "COFORGE.NS", "IDFCFIRSTB.NS", "ASHOKLEY.NS", "MRF.NS", "PIIND.NS", "CONCOR.NS", "JUBLFOOD.NS", "VOLTAS.NS", "OBEROIRLTY.NS", "GODREJPROP.NS", "AUBANK.NS", "ABCAPITAL.NS", "APOLLOTYRE.NS", "BALKRISIND.NS", "BANDHANBNK.NS", "BANKINDIA.NS", "BHARATFORG.NS", "CANBK.NS", "CHOLAFIN.NS", "COROMANDEL.NS", "CROMPTON.NS", "DALBHARAT.NS", "ESCORTS.NS", "EXIDEIND.NS", "GMRINFRA.NS", "GUJGASLTD.NS", "HAL.NS", "HINDPETRO.NS", "IGL.NS", "INDIANB.NS", "INDHOTEL.NS", "IPCALAB.NS", "JSWENERGY.NS", "L&TFH.NS", "LICHSGFIN.NS", "MAXHEALTH.NS", "MFSL.NS", "MGL.NS", "MUTHOOTFIN.NS", "NAUKRI.NS", "PAGEIND.NS", "SUZLON.NS", "CDSL.NS", "ANGELONE.NS", "RADICO.NS", "PNBHOUSING.NS", "CYIENT.NS", "KALYANKJIL.NS", "CAMS.NS", "SONACOMS.NS", "KPITTECH.NS", "UTIAMC.NS", "RVNL.NS", "IRFC.NS", "MAZDOCK.NS", "COCHINSHIP.NS", "RAYMOND.NS", "BEML.NS", "CENTURYTEX.NS", "CESC.NS", "CHAMBLFERT.NS", "EQUITASBNK.NS", "GLENMARK.NS", "GNFC.NS", "GRANULES.NS", "GSPL.NS", "HFCL.NS", "HINDCOPPER.NS", "JBCHEPHARM.NS", "JINDALSAW.NS", "LATENTVIEW.NS", "LXCHEM.NS", "MCX.NS", "METROPOLIS.NS", "NATCOPHARM.NS", "NBCC.NS", "PRAJIND.NS", "PRINCEPIPE.NS", "PVRINOX.NS", "REDINGTON.NS", "RENUKA.NS", "ROUTE.NS", "SYNGENE.NS", "TEJASNET.NS", "TIDEL.NS", "TRIDENT.NS", "UCOBANK.NS", "VIPIND.NS", "WELCORP.NS", "ZENSARTECH.NS", "ZYDUSWELL.NS"
]
logger.info(f"Successfully loaded {len(NIFTY_UNIVERSE)} stratified stocks!")


In [ ]:
logger.info("Downloading 5 years of evolutionary data and NIFTY 50 Benchmark...")
raw = yf.download(NIFTY_UNIVERSE, period="5y", interval="1d", auto_adjust=False, progress=False)
benchmark_raw = yf.download("^NSEI", period="5y", interval="1d", auto_adjust=False, progress=False)

all_close = raw.iloc[:, raw.columns.get_level_values(0) == 'Close'].copy()
all_close.columns = all_close.columns.droplevel(0)

all_high = raw.iloc[:, raw.columns.get_level_values(0) == 'High'].copy()
all_high.columns = all_high.columns.droplevel(0)

all_low = raw.iloc[:, raw.columns.get_level_values(0) == 'Low'].copy()
all_low.columns = all_low.columns.droplevel(0)

all_vol = raw.iloc[:, raw.columns.get_level_values(0) == 'Volume'].copy()
all_vol.columns = all_vol.columns.droplevel(0)

# CRITICAL FIX 1: Use actual NIFTY 50 instead of a survivorship-biased mean
market_bench = benchmark_raw['Close'].squeeze()
nifty_fwd_ret_global = market_bench.shift(-5) / market_bench - 1

logger.info("Data successfully downloaded.")

In [ ]:
all_dfs = []
logger.info("Preparing raw base features for the AI to mutate...")

for sym in NIFTY_UNIVERSE:
    if sym not in all_close.columns:
        continue
        
    df = pd.DataFrame({
        'close': all_close[sym],
        'high': all_high[sym],
        'low': all_low[sym],
        'volume': all_vol[sym]
    }).dropna()
    
    if len(df) < 252: continue
        
    # Base Ingredients for the Alpha Factory
    df['ret_1d'] = df['close'].pct_change(1)
    df['ret_3d'] = df['close'].pct_change(3)
    df['ret_5d'] = df['close'].pct_change(5)
    df['ret_10d'] = df['close'].pct_change(10)
    df['ret_20d'] = df['close'].pct_change(20)
    
    # Moving Average Distances
    df['ma_5_dist'] = df['close'] / (df['close'].rolling(5).mean() + 1e-9) - 1
    df['ma_20_dist'] = df['close'] / (df['close'].rolling(20).mean() + 1e-9) - 1
    
    # Volatility and Spreads
    df['vol_norm_5d'] = df['volume'] / (df['volume'].rolling(5).mean() + 1e-9)
    df['vol_norm_20d'] = df['volume'] / (df['volume'].rolling(20).mean() + 1e-9)
    df['high_low_spread'] = (df['high'] - df['low']) / df['close']
    df['close_pos'] = (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-9)
    # The Ultimate Goal (Target)
    fwd_ret = df['close'].shift(-5) / df['close'] - 1
    
    # Reindex safely aligning dates
    nifty_aligned = nifty_fwd_ret_global.reindex(df.index)
    
    # We want it to find formulas that positively correlate with BEATING the market
    df['target'] = (fwd_ret - nifty_aligned).fillna(0)
    df['symbol'] = sym
    
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    all_dfs.append(df)

dataset = pd.concat(all_dfs)

# CRITICAL FIX 2: Sort strictly by Date to allow a true chronological split
dataset = dataset.sort_index()

logger.info(f"Evolution Arena Ready: {len(dataset)} historical trading days loaded.")

In [ ]:
# CRITICAL FIX 3: Out-Of-Sample Chronological Split (Walk-Forward Testing style)
# Train on the first 80% of time, test the genetics on the unseen future 20%
split_idx = int(len(dataset) * 0.8)
train_data = dataset.iloc[:split_idx]
test_data = dataset.iloc[split_idx:]

feature_cols = ['ret_1d', 'ret_3d', 'ret_5d', 'ret_10d', 'ret_20d', 'ma_5_dist', 'ma_20_dist', 'vol_norm_5d', 'vol_norm_20d', 'high_low_spread', 'close_pos']
X_train = train_data[feature_cols].values
y_train = train_data['target'].values

logger.info(f"Training Set: {len(X_train)} samples. Testing Set (OOS): {len(test_data)} samples.")
logger.info("🧬 INITIATING GENETIC EVOLUTION (This will take time on CPU/GPU)...")
logger.info("Spawning 1000 random mathematical formulas per generation...")

# The Alpha Factory Settings
function_set = ['add', 'sub', 'mul', 'div', 'sqrt', 'log', 'abs', 'neg', 'inv', 'max', 'min']

gp = SymbolicTransformer(
    generations=60,            
    population_size=8000,      
    hall_of_fame=100,          
    n_components=3,            # Return the absolute best 3 Super Formulas
    function_set=function_set, 
    metric='spearman',          # Optimizing for linear correlation to the alpha target
    parsimony_coefficient=0.005, # Punish formulas that become too long/overfitted
    max_samples=0.9,           # Randomly sample within the chronological training set to boost variety
    verbose=1, 
    random_state=42, 
    n_jobs=-1                  
)

gp.fit(X_train, y_train)
logger.info("Evolution Complete!")

In [ ]:
print("\n======================================================")
print("🏆 SURVIVAL OF THE FITTEST: THE 3 WINNING SUPER-FORMULAS 🏆")
print("======================================================\n")

# We transform the untouched OOS test data using the evolved formulas to ensure they aren't noise
X_test = test_data[feature_cols].values
y_test = test_data['target'].values
transformed_test = gp.transform(X_test)

output_log = []
for i, formula in enumerate(gp): 
    res_str = f"Super Formula {i+1}:\n{formula}\n"
    
    # Check Rank Correlation (IC - Information Coefficient) on the Out-Of-Sample Data
    formula_values = transformed_test[:, i]
    ic_score = pd.Series(formula_values).corr(pd.Series(y_test), method='spearman')
    res_str += f"Out-Of-Sample Rank Correlation (IC): {ic_score:.4f}\n"
    
    print(res_str)
    output_log.append(res_str)
    
print("======================================================")
print("To implement these in your live bot:")
print("1. Ensure the Out-Of-Sample IC score is solid (e.g., > 0.02 or < -0.02).")
print("2. Map X0 (ret_1d), X3 (vol_norm), etc., and write it in Python in src/screening/ml_factor.py.")
print("3. Retrain your XGBoost champion model to evaluate true contribution.")
print("======================================================")

# CRITICAL KAGGLE FIX: Save to a file so you don't lose the formulas when you close your browser!
with open("winning_formulas.txt", "w") as f:
    f.write("\n".join(output_log))
print("✅ Saved formulas to winning_formulas.txt for easy download.")
